# AlphaFlow / ESMFlow — second-generator cross-check on the T4L benchmark

Generates a T4L L99A ensemble with **ESMFlow** (the no-MSA, ESMFold-based AlphaFlow variant) and scores it against the SAME frozen 2LC9 excited-state CV used for BioEmu, so the numbers are directly comparable.

**Setup:** Runtime → Change runtime type → **GPU (A100)**. Run cells top to bottom.

**Install recipe:** uses the *confirmed CUDA-12 config* from AlphaFlow issue #40 (torch **2.5.1** + OpenFold **`pl_upgrades`** branch + py3.9) — NOT the README's outdated torch-1.12/cu113 pins, which fight Colab's CUDA 12. We build it in an isolated conda env via `condacolab`.

**The one fragile step is Cell 2** (OpenFold compiles CUDA kernels). **Cell 3 verifies the env before you spend any GPU.** If Cell 2/3 fail, paste the error. Generation runs in the `af` env; scoring runs in base via this repo + MDAnalysis — they only talk through PDB files on disk.

In [ ]:
# 1) Install conda on Colab (this AUTO-RESTARTS the runtime). After it restarts, run Cell 2.
!pip install -q condacolab
import condacolab; condacolab.install()

In [ ]:
# 2) Build the AlphaFlow env -- CONFIRMED CUDA-12 recipe (AlphaFlow issue #40):
#    py3.9 + torch 2.5.1 (matches Colab's CUDA 12) + OpenFold 'pl_upgrades' branch.
#    ~10-15 min; the openfold 'pip install -e .' compiles CUDA kernels (the fragile step).
%%bash
set -e
cd /content
[ -d alphaflow ] || git clone -q https://github.com/bjing2016/alphaflow.git
[ -d openfold ]  || git clone -q -b pl_upgrades https://github.com/aqlaboratory/openfold.git
conda create -y -n af python=3.9 >/dev/null
source activate af
export CUDA_HOME=/usr/local/cuda
pip install -q ninja numpy==1.21.2 pandas==1.5.3
pip install -q torch==2.5.1
pip install -q biopython dm-tree==0.1.6 modelcif==0.7 ml-collections==0.1.0 scipy==1.7.1 absl-py einops
pip install -q pytorch_lightning==2.0.4 fair-esm mdtraj==1.9.9 wandb
cd /content/openfold && pip install -q -e .
echo 'ENV BUILD DONE'

In [ ]:
# 3) VERIFY the env imports BEFORE spending GPU. Must print torch + cuda True and 'OK'.
%%bash
source activate af
python - <<'PY'
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
import openfold, esm, pytorch_lightning  # the imports AlphaFlow needs
print('OK: openfold/esm/lightning import')
PY

In [ ]:
# 4) Download the ESMFlow-MD distilled weights (no MSA needed; distilled = fast).
%%bash
mkdir -p /content/weights
cd /content/weights
[ -f esmflow_md_distilled.pt ] || wget -q -O esmflow_md_distilled.pt \
  https://huggingface.co/bjing-mit/alphaflow/resolve/main/params/esmflow_md_distilled_202402.pt
ls -la /content/weights

In [ ]:
# 5) Write the AlphaFlow input CSV (name,seqres) for T4L L99A.
import os
SEQ = ("MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAIGRNTNGVITKDEAEKLFNQ"
       "DVDAAVRGILRNAKLKPVYDSLDAVRRAAAINMVFQMGETGVAGFTNSLRMLQQKRWDEAAVNLAKSRW"
       "YNQTPNRAKRVITTFRTGTWDAYK")
os.makedirs('/content/af_in', exist_ok=True)
with open('/content/af_in/t4l.csv', 'w') as f:
    f.write('name,seqres\n')
    f.write('t4l,' + SEQ + '\n')
print('wrote /content/af_in/t4l.csv  (', len(SEQ), 'residues )')

In [ ]:
# 6) Generate the ensemble (ESMFlow distilled). 256 samples is plenty for rare-state recall.
#    Output -> outputs/benchmark/generated/t4l/alphaflow/seed0  (matches the scorer's layout).
%%bash
source activate af
export CUDA_HOME=/usr/local/cuda
mkdir -p /content/outputs/benchmark/generated/t4l/alphaflow/seed0
cd /content/alphaflow
python predict.py --mode esmfold \
  --input_csv /content/af_in/t4l.csv \
  --weights /content/weights/esmflow_md_distilled.pt \
  --samples 256 --noisy_first --no_diffusion \
  --outpdb /content/outputs/benchmark/generated/t4l/alphaflow/seed0
echo '--- output files ---'
ls -la /content/outputs/benchmark/generated/t4l/alphaflow/seed0 | head

In [ ]:
# 7) Score with the SAME frozen CV used for BioEmu (runs in base env via this repo + MDAnalysis).
import os, subprocess, sys
os.chdir('/content')
if not os.path.isdir('/content/conformational-ml/.git'):
    subprocess.run(['git','clone','-q','https://github.com/realakshayk1/conformational-ml'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','MDAnalysis','pyyaml'], check=True)
subprocess.run([sys.executable,'scripts/fetch_ground_truth_structures.py'], cwd='/content/conformational-ml', check=True)
subprocess.run([sys.executable,'scripts/run_benchmark.py',
    '--generated-dir','/content/outputs/benchmark/generated/t4l/alphaflow',
    '--config','configs/evaluation/t4l_excited_state.yaml',
    '--label','alphaflow_t4l',
    '--out','/content/outputs/benchmark/t4l_benchmark_alphaflow.json',
    '--records-out','/content/outputs/benchmark/t4l_records_alphaflow.json'],
    cwd='/content/conformational-ml', check=True)

In [ ]:
# 8) Figure + headline, and a side-by-side with BioEmu if its JSON is present.
#    (Fresh runtime: drag your saved t4l_benchmark_bioemu.json into /content/outputs/benchmark/ first.)
import json, numpy as np, yaml, matplotlib.pyplot as plt, os
CONFIG = '/content/conformational-ml/configs/evaluation/t4l_excited_state.yaml'
OUT = '/content/outputs/benchmark/t4l_benchmark_alphaflow.json'
REC = '/content/outputs/benchmark/t4l_records_alphaflow.json'
summary = json.load(open(OUT)); records = json.load(open(REC))
cfg = yaml.safe_load(open(CONFIG))
margin = cfg['contrast']['margin_angstrom']; exp = cfg.get('experimental_target_population')
deltas = np.array([r['delta'] for r in records]); o = summary['overall']

plt.figure(figsize=(7,4))
plt.hist(deltas, bins=40, color='#C44E52', alpha=0.85)
plt.axvline(margin, color='black', ls='--', label=f'target threshold (\u0394 > {margin})')
plt.xlabel('\u0394 = RMSD_to_ground \u2212 RMSD_to_target (\u00c5)')
plt.ylabel('conformers'); plt.title(f"T4L AlphaFlow/ESMFlow  (n={o['n_conformers']})")
plt.legend(); plt.tight_layout()
plt.savefig('/content/outputs/benchmark/t4l_alphaflow_hist.png', dpi=150); plt.show()

print(f"ALPHAFLOW  target-state-like: {o['frac_target_state_like']*100:.2f}%  (experimental ~{exp*100:.0f}%)")
print(f"           min RMSD to target: {o['min_rmsd_to_target']:.2f} A   geometry valid: {o['geometry_validity_rate']*100:.1f}%")
bio = '/content/outputs/benchmark/t4l_benchmark_bioemu.json'
if os.path.exists(bio):
    b = json.load(open(bio))['overall']
    print('\n--- T4L cross-method comparison (vs experimental 3%) ---')
    print(f"  BioEmu    : {b['frac_target_state_like']*100:5.2f}%  (min RMSD {b['min_rmsd_to_target']:.2f} A)")
    print(f"  AlphaFlow : {o['frac_target_state_like']*100:5.2f}%  (min RMSD {o['min_rmsd_to_target']:.2f} A)")